# T05: LinkML Schema Export

This tutorial generates a unified LinkML YAML schema from all ingested backend
elements and verifies it is importable via `LinkMLAdapter`.

**Services required**: backend (`http://localhost:8002`)

**Requires**: undata CLI (`cd ../ingestion && uv sync`)

**Est. time**: 5 min

In [1]:
# Cell 2 — service availability check
import os
import subprocess
from pathlib import Path

import httpx

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8002")
API_KEY = os.getenv(
    "API_KEY",
    "qs005testtoken1234567890abcdef1234567890abcdef1234567890abcdef12",
)
HEADERS = {"Authorization": f"Bearer {API_KEY}"}
INGESTION_DIR = os.getenv(
    "INGESTION_DIR",
    str(Path("../ingestion").resolve()),
)
OUTPUT_PATH = "/tmp/undata-tutorial-schema.yaml"

try:
    httpx.get(f"{BACKEND_URL}/health", timeout=2.0).raise_for_status()
    print(f"✓ Backend available at {BACKEND_URL}")
except Exception as _e:
    import pytest

    pytest.skip(f"Backend unavailable: {_e}")

✓ Backend available at http://localhost:8002


## 1. Generate Unified Schema via CLI

The `undata generate-schema` command queries the backend for all elements and
generates a LinkML YAML schema that represents their types and relationships.

In [2]:
result = subprocess.run(
    [
        "uv",
        "run",
        "undata",
        "generate-schema",
        "--output",
        OUTPUT_PATH,
        "--backend-url",
        f"{BACKEND_URL}/api/v1",
    ],
    cwd=INGESTION_DIR,
    capture_output=True,
    text=True,
)
print("STDOUT:", result.stdout[-1000:] if result.stdout else "(none)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
assert result.returncode == 0, f"generate-schema failed with code {result.returncode}"
assert Path(OUTPUT_PATH).exists(), f"Output file not found: {OUTPUT_PATH}"
size = Path(OUTPUT_PATH).stat().st_size
print(f"✓ Schema written to {OUTPUT_PATH} ({size} bytes)")

STDOUT: (none)
✓ Schema written to /tmp/undata-tutorial-schema.yaml (218664 bytes)


## 2. Inspect the Generated Schema

Load the generated YAML with `LinkMLAdapter` to inspect elements and classes.

In [3]:
_inspect_result = subprocess.run(
    [
        "uv",
        "run",
        "python",
        "-c",
        f"""
from undata.adapters.linkml_adapter import LinkMLAdapter
la = LinkMLAdapter()
la.load_file({OUTPUT_PATH!r})
elements = la.extract_elements()
classes = la.extract_classes()
print(f'Elements: {{len(elements)}}, Classes: {{len(classes)}}')
print('\\nFirst 5 slot names:')
for el in elements[:5]:
    print(f'  - {{el.name}} | data_type={{el.data_type}}')
""",
    ],
    cwd=INGESTION_DIR,
    capture_output=True,
    text=True,
)
print(_inspect_result.stdout)
if _inspect_result.returncode != 0:
    print("STDERR:", _inspect_result.stderr[-500:])
assert _inspect_result.returncode == 0, f"LinkMLAdapter failed: {_inspect_result.returncode}"

Elements: 1088, Classes: 21

First 5 slot names:
  - url | data_type=string
  - name | data_type=string
  - schemaKey | data_type=string
  - wasAssociatedWith | data_type=string
  - identifier | data_type=string



## 3. View Schema Metadata via API

The backend tracks all dynamic schemas. The `/api/v1/schemas/` endpoint lists them.

In [4]:
response = httpx.get(
    f"{BACKEND_URL}/api/v1/schemas/",
    headers=HEADERS,
    params={"limit": 5},
    timeout=5.0,
)
assert response.status_code == 200, f"Expected 200, got {response.status_code}"
data = response.json()
schemas = data if isinstance(data, list) else data.get("items", [])
print(f"Registered schemas: {len(schemas)}")
for schema in schemas:
    print(f"  - {schema.get('name', '?')} (created: {schema.get('created_at', '?')[:10]})")

Registered schemas: 5
  - PerfChild1773236867 (created: 2026-03-11)
  - PerfParent1773236867 (created: 2026-03-11)
  - PerfGrand1773236867 (created: 2026-03-11)
  - PerfChild1773236575 (created: 2026-03-11)
  - PerfParent1773236575 (created: 2026-03-11)


## Cleanup

In [5]:
if Path(OUTPUT_PATH).exists():
    os.unlink(OUTPUT_PATH)
    print(f"✓ Deleted {OUTPUT_PATH}")
print("✓ Cleanup complete")

✓ Deleted /tmp/undata-tutorial-schema.yaml
✓ Cleanup complete


## Next Steps

You've generated a unified LinkML schema and confirmed it can be re-imported.

Next: **[T06: Schema Roundtrip Validation](06_schema_roundtrip.ipynb)** — offline
validation of custom schemas using the ingestion library directly.